# 멀티 뷰 구성 (Multi-View Composition)

여러 다른 데이터 필드를 시각화할 때 가능한 한 많은 시각적 인코딩 채널(`x`, `y`, `color`, `size`, `shape` 등)을 사용하고 싶은 유혹을 느낄 수 있습니다. 그러나 인코딩 채널의 수가 늘어남에 따라 차트는 빠르게 복잡해지고 읽기 어려워질 수 있습니다. 단일 차트에 "과부하"를 주는 대신, 빠른 비교를 용이하게 하는 방식으로 *여러 차트를 구성(compose multiple charts)*하는 것이 대안이 될 수 있습니다.

이 노트북에서는 다음과 같은 다양한 *멀티 뷰 구성* 작업을 살펴보겠습니다:

- *레이어(layer)*: 호환되는 차트를 서로 바로 위에 배치합니다.
- *패싯(facet)*: 데이터를 행 또는 열로 구성된 여러 차트로 분할합니다.
- *연결(concatenate)*: 공유된 레이아웃 내에 임의의 차트를 배치합니다.
- *반복(repeat)*: 기본 차트 사양을 가져와 여러 데이터 필드에 적용합니다.

그런 다음 이러한 작업이 어떻게 *뷰 구성 대수(view composition algebra)*를 형성하는지 살펴볼 것입니다. 여기에서 작업을 결합하여 다양하고 복잡한 멀티 뷰 디스플레이를 구축할 수 있습니다.

_이 노트북은 [데이터 시각화 커리큘럼](https://github.com/uwdata/visualization-curriculum)의 일부입니다._

In [1]:
import pandas as pd
import altair as alt

## 날씨 데이터 (Weather Data)

우리는 미국의 도시인 시애틀과 뉴욕의 날씨 통계를 시각화할 것입니다. 데이터 세트를 로드하고 처음과 마지막 10행을 살펴보겠습니다:

In [2]:
weather = 'https://cdn.jsdelivr.net/npm/vega-datasets@1/data/weather.csv'

In [3]:
df = pd.read_csv(weather)
df.head(10)

,location,date,precipitation,temp_max,temp_min,wind,weather
0,Seattle,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,Seattle,2012-01-02,10.9,10.6,2.8,4.5,rain
2,Seattle,2012-01-03,0.8,11.7,7.2,2.3,rain
3,Seattle,2012-01-04,20.3,12.2,5.6,4.7,rain
4,Seattle,2012-01-05,1.3,8.9,2.8,6.1,rain
5,Seattle,2012-01-06,2.5,4.4,2.2,2.2,rain
6,Seattle,2012-01-07,0.0,7.2,2.8,2.3,rain
7,Seattle,2012-01-08,0.0,10.0,2.8,2.0,sun
8,Seattle,2012-01-09,4.3,9.4,5.0,3.4,rain
9,Seattle,2012-01-10,1.0,6.1,0.6,3.4,rain


In [4]:
df.tail(10)

,location,date,precipitation,temp_max,temp_min,wind,weather
2912,New York,2015-12-22,4.8,15.6,11.1,3.8,fog
2913,New York,2015-12-23,29.5,17.2,8.9,4.5,fog
2914,New York,2015-12-24,0.5,20.6,13.9,4.9,fog
2915,New York,2015-12-25,2.5,17.8,11.1,0.9,fog
2916,New York,2015-12-26,0.3,15.6,9.4,4.8,drizzle
2917,New York,2015-12-27,2.0,17.2,8.9,5.5,fog
2918,New York,2015-12-28,1.3,8.9,1.7,6.3,snow
2919,New York,2015-12-29,16.8,9.4,1.1,5.3,fog
2920,New York,2015-12-30,9.4,10.6,5.0,3.0,fog
2921,New York,2015-12-31,1.5,11.1,6.1,5.5,fog


우리는 도시 내의 날씨와 도시 간의 날씨를 조사하기 위해 멀티 뷰 디스플레이를 만들 것입니다.

## 레이어 (Layer)

여러 차트를 결합하는장 일반적인 방법 중 하나는 마크를 서로 위에 *레이어(layer)*로 쌓는 것입니다. 기초가 되는 스케일 도메인이 호환되는 경우, 이를 병합하여 *공유 축(shared axes)*을 형성할 수 있습니다. `x` 또는 `y` 인코딩 중 하나라도 호환되지 않는 경우, 대신 별도의 스케일과 축을 사용하여 마크를 겹치는 *이중 축 차트(dual-axis chart)*를 만들 수 있습니다.

### 공유 축 (Shared Axes)

먼저 월별 최소 및 최대 평균 기온을 플로팅해 보겠습니다:

In [5]:
alt.Chart(weather).mark_area().encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_max):Q'),
  alt.Y2('average(temp_min):Q')
)

alt.Chart(...)

*이 플롯은 데이터 전체 기간 동안 각 월의 기온 범위를 보여줍니다. 그러나 시애틀과 뉴욕의 측정값을 모두 합산하기 때문에 다소 오해의 소지가 있습니다!*

색상 인코딩을 사용하여 위치별로 데이터를 세분화하고, 겹치는 영역을 고려하여 마크 불투명도(opacity)를 조정해 보겠습니다:

In [6]:
alt.Chart(weather).mark_area(opacity=0.3).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_max):Q'),
  alt.Y2('average(temp_min):Q'),
  alt.Color('location:N')
)

alt.Chart(...)

*시애틀이 더 온화하다는 것을 알 수 있습니다: 겨울에는 더 따뜻하고 여름에는 더 시원합니다.*

이 경우 단순히 영역 마크를 색상별로 세분화하여 특별한 기능 없이 레이어 차트를 만들었습니다. 위의 차트는 기온 범위를 보여주지만, 범위의 중간 지점을 강조하고 싶을 수도 있습니다.

평균 기온의 중간 지점을 보여주는 선 차트를 만들어 보겠습니다. 일일 최저 기온과 최고 기온 사이의 중간 지점을 계산하기 위해 `calculate` 변환을 사용하겠습니다:

In [7]:
alt.Chart(weather).mark_line().transform_calculate(
  temp_mid='(+datum.temp_min + +datum.temp_max) / 2'
).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_mid):Q'),
  alt.Color('location:N')
)

alt.Chart(...)

*참고*: calculate 변환 내에서 `+datum.temp_min`을 사용한 것에 주목하세요. 특별한 파싱 지침 없이 CSV 파일에서 데이터를 직접 로드하기 때문에 기온 값이 내부적으로 문자열 값으로 표현될 수 있습니다. 값 앞에 `+`를 추가하면 숫자로 처리되도록 강제합니다.

이제 범위 영역 위에 중간 지점 선을 레이어로 쌓아 이 차트들을 결합해 보겠습니다. `chart1 + chart2` 구문을 사용하여 `chart1`이 첫 번째 레이어이고 `chart2`가 그 위에 그려지는 두 번째 레이어인 새로운 레이어 차트를 지정할 수 있습니다:

In [8]:
tempMinMax = alt.Chart(weather).mark_area(opacity=0.3).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_max):Q'),
  alt.Y2('average(temp_min):Q'),
  alt.Color('location:N')
)

tempMid = alt.Chart(weather).mark_line().transform_calculate(
  temp_mid='(+datum.temp_min + +datum.temp_max) / 2'
).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_mid):Q'),
  alt.Color('location:N')
)

tempMinMax + tempMid

alt.LayerChart(...)

*이제 멀티 레이어 플롯이 완성되었습니다! 하지만 y축 제목이 (정보를 담고는 있지만) 다소 길고 정리가 안 된 느낌입니다...*

플롯을 정리하기 위해 축을 사용자 정의해 보겠습니다. 레이어 중 하나에서 사용자 정의 축 제목을 설정하면 모든 레이어에 대한 공유 축 제목으로 자동으로 사용됩니다:

In [9]:
tempMinMax = alt.Chart(weather).mark_area(opacity=0.3).encode(
  alt.X('month(date):T', title=None, axis=alt.Axis(format='%b')),
  alt.Y('average(temp_max):Q', title='평균 최고 기온 °C'),
  alt.Y2('average(temp_min):Q'),
  alt.Color('location:N')
)

tempMid = alt.Chart(weather).mark_line().transform_calculate(
  temp_mid='(+datum.temp_min + +datum.temp_max) / 2'
).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_mid):Q'),
  alt.Color('location:N')
)

tempMinMax + tempMid

alt.LayerChart(...)

*두 레이어 모두에 사용자 정의 축 제목이 있으면 어떻게 될까요? 위의 코드를 수정하여 확인해 보세요...*

위에서는 Altair의 `layer` 메서드에 대한 편리한 약칭인 `+` 연산자를 사용했습니다. `layer` 메서드를 직접 사용하여 동일한 레이어 차트를 생성할 수 있습니다:

In [10]:
alt.layer(tempMinMax, tempMid)

alt.LayerChart(...)

레이어에 대한 입력 순서가 중요하다는 점에 유의하세요. 후속 레이어가 이전 레이어 위에 그려지기 때문입니다. *위의 셀에서 차트의 순서를 바꿔보세요. 어떤 일이 일어나나요? (힌트: `line` 마크의 색상을 주의 깊게 살펴보세요.)*

### 이중 축 차트 (Dual-Axis Charts)

*시애틀은 비가 많이 오는 도시라는 평판이 있습니다. 그럴 만한 이유가 있을까요?*

더 알아보기 위해 기온과 함께 강수량을 살펴보겠습니다. 먼저 시애틀의 월평균 강수량을 보여주는 기본 플롯을 만들어 보겠습니다:

In [11]:
alt.Chart(weather).transform_filter(
  'datum.location == "Seattle"'
).mark_line(
  interpolate='monotone',
  stroke='grey'
).encode(
  alt.X('month(date):T', title=None),
  alt.Y('average(precipitation):Q', title='Precipitation')
)

alt.Chart(...)

기온 데이터와 쉽게 비교할 수 있도록 새로운 레이어 차트를 만들어 보겠습니다. 이전과 같이 차트를 레이어로 쌓으려고 하면 어떤 일이 일어나는지 확인해 보세요:

In [12]:
tempMinMax = alt.Chart(weather).transform_filter(
  'datum.location == "Seattle"'
).mark_area(opacity=0.3).encode(
  alt.X('month(date):T', title=None, axis=alt.Axis(format='%b')),
  alt.Y('average(temp_max):Q', title='평균 최고 기온 °C'),
  alt.Y2('average(temp_min):Q')
)

precip = alt.Chart(weather).transform_filter(
  'datum.location == "Seattle"'
).mark_line(
  interpolate='monotone',
  stroke='grey'
).encode(
  alt.X('month(date):T'),
  alt.Y('average(precipitation):Q', title='Precipitation')
)

alt.layer(tempMinMax, precip)

alt.LayerChart(...)

*강수량 값은 기온보다 y축의 훨씬 좁은 범위를 사용합니다!*

기본적으로 레이어 차트는 *공유 도메인(shared domain)*을 사용합니다. 즉, x축 또는 y축의 값이 모든 레이어에서 결합되어 공유된 범위(extent)를 결정합니다. 이 기본 동작은 레이어로 쌓인 값들이 동일한 단위를 갖는다고 가정합니다. 그러나 이 예제에서는 기온 값(섭씨)과 강수량 값(인치)을 결합하고 있으므로 이 가정이 맞지 않습니다!

서로 다른 y축 스케일을 사용하려면 Altair가 레이어 전체에서 데이터를 어떻게 *해결(resolve)*하기를 원하는지 지정해야 합니다. 이 경우 y축 `scale` 도메인이 `shared`(공유) 도메인을 사용하는 대신 `independent`(독립적)가 되도록 해결하고자 합니다. 레이어 연산자에 의해 생성된 `Chart` 객체에는 원하는 해결 방식을 지정할 수 있는 `resolve_scale` 메서드가 포함되어 있습니다:

In [13]:
tempMinMax = alt.Chart(weather).transform_filter(
  'datum.location == "Seattle"'
).mark_area(opacity=0.3).encode(
  alt.X('month(date):T', title=None, axis=alt.Axis(format='%b')),
  alt.Y('average(temp_max):Q', title='평균 최고 기온 °C'),
  alt.Y2('average(temp_min):Q')
)

precip = alt.Chart(weather).transform_filter(
  'datum.location == "Seattle"'
).mark_line(
  interpolate='monotone',
  stroke='grey'
).encode(
  alt.X('month(date):T'),
  alt.Y('average(precipitation):Q', title='Precipitation')
)

alt.layer(tempMinMax, precip).resolve_scale(y='independent')

alt.LayerChart(...)

*이제 시애틀에서 가을이 가장 비가 많이 오는 계절(11월에 최고조)이며, 건조한 여름이 이를 보완한다는 것을 알 수 있습니다.*

위의 플롯 사양에서 일부 중복을 발견하셨을 것입니다. 두 차트 모두 동일한 데이터 세트와 동일한 필터를 사용하여 시애틀만 살펴봅니다. 원한다면 최상위 레이어 차트에 데이터와 필터 변환을 제공하여 코드를 약간 간소화할 수 있습니다. 그러면 개별 레이어는 자체 데이터 정의가 없는 경우 데이터를 상속받게 됩니다:

In [14]:
tempMinMax = alt.Chart().mark_area(opacity=0.3).encode(
  alt.X('month(date):T', title=None, axis=alt.Axis(format='%b')),
  alt.Y('average(temp_max):Q', title='평균 최고 기온 °C'),
  alt.Y2('average(temp_min):Q')
)

precip = alt.Chart().mark_line(
  interpolate='monotone',
  stroke='grey'
).encode(
  alt.X('month(date):T'),
  alt.Y('average(precipitation):Q', title='Precipitation')
)

alt.layer(tempMinMax, precip, data=weather).transform_filter(
  'datum.location == "Seattle"'
).resolve_scale(y='independent')

alt.LayerChart(...)

이중 축 차트가 유용할 수 있지만, 서로 다른 단위와 축 스케일이 비례하지 않을 수 있기 때문에 *종종 오해를 불러일으키기 쉽습니다*. 가능한 경우 서로 다른 데이터 필드를 공유 단위로 매핑하는 변환(예: [분위수(quantiles)](https://ko.wikipedia.org/wiki/%EB%B6%84%EC%9C%84%EC%88%98) 또는 상대적 백분율 변화를 보여주는 방식)을 고려할 수 있습니다.

## 패싯 (Facet)

*패싯(Faceting)*은 데이터 세트를 그룹으로 세분화하고 각 그룹에 대해 별도의 플롯을 생성하는 것을 포함합니다. 이전 노트북에서 `row` 및 `column` 인코딩 채널을 사용하여 패싯 차트를 만드는 방법을 배웠습니다. 먼저 해당 채널들을 복습한 다음, 그것들이 어떻게 더 일반적인 `facet` 연산자의 사례인지 보여드리겠습니다.

시애틀의 최고 기온 값에 대한 기본 히스토그램부터 시작하겠습니다:

In [15]:
alt.Chart(weather).mark_bar().transform_filter(
  'datum.location == "Seattle"'
).encode(
  alt.X('temp_max:Q', bin=True, title='최고 기온 (°C)'),
  alt.Y('count():Q')
)

alt.Chart(...)

*이 기온 프로필이 이슬비, 안개, 비, 눈, 태양 등 주어진 날의 날씨에 따라 어떻게 변할까요?*

`column` 인코딩 채널을 사용하여 날씨 유형별로 데이터를 패싯해 보겠습니다. 또한 사용자 정의된 색상 범위를 사용하여 `color`를 중복 인코딩으로 사용할 수 있습니다:

In [16]:
colors = alt.Scale(
  domain=['drizzle', 'fog', 'rain', 'snow', 'sun'],
  range=['#aec7e8', '#c7c7c7', '#1f77b4', '#9467bd', '#e7ba52']
)

alt.Chart(weather).mark_bar().transform_filter(
  'datum.location == "Seattle"'
).encode(
  alt.X('temp_max:Q', bin=True, title='최고 기온 (°C)'),
  alt.Y('count():Q'),
  alt.Color('weather:N', scale=colors),
  alt.Column('weather:N')
).properties(
  width=150,
  height=150
)

alt.Chart(...)

*당연하게도 드문 눈이 오는 날은 가장 추운 기온에 집중되어 있으며, 비가 오고 안개가 낀 날이 그 뒤를 잇습니다. 맑은 날은 더 따뜻하며, 시애틀에 대한 고정관념에도 불구하고 가장 많습니다. 하지만 시애틀 사람이라면 누구나 말해주듯, 기온에 상관없이 이슬비는 가끔 내립니다!*

차트 정의 *내*의 `row` 및 `column` 인코딩 채널 외에도, 기본 차트 정의를 가져와 명시적인 `facet` 연산자를 사용하여 패싯을 적용할 수 있습니다.

위의 차트를 다시 만들어 보되, 이번에는 `facet`을 사용해 보겠습니다. 동일한 기본 히스토그램 정의로 시작하지만 데이터 소스, 필터 변환 및 컬럼 채널을 제거합니다. 그런 다음 `facet` 메서드를 호출하여 데이터를 전달하고 `weather` 필드에 따라 열로 패싯하도록 지정합니다. `facet` 메서드는 `row`와 `column` 인수를 모두 허용합니다. 두 인수를 함께 사용하여 패싯 플롯의 2D 그리드를 만들 수 있습니다.

마지막으로 필터 변환을 포함하여 최상위 패싯 차트에 적용합니다. 이전처럼 히스토그램 정의에 필터 변환을 적용할 수도 있지만, 그것은 약간 덜 효율적입니다. 각 패싯 셀 내에서 "New York" 값을 걸러내는 대신, 패싯된 차트에 필터를 적용하면 Vega-Lite가 패싯 세분화 전에 미리 해당 값을 걸러낼 수 있음을 알게 됩니다.

In [17]:
colors = alt.Scale(
  domain=['drizzle', 'fog', 'rain', 'snow', 'sun'],
  range=['#aec7e8', '#c7c7c7', '#1f77b4', '#9467bd', '#e7ba52']
)

alt.Chart().mark_bar().encode(
  alt.X('temp_max:Q', bin=True, title='최고 기온 (°C)'),
  alt.Y('count():Q'),
  alt.Color('weather:N', scale=colors)
).properties(
  width=150,
  height=150
).facet(
  data=weather,
  column='weather:N'
).transform_filter(
  'datum.location == "Seattle"'
)

alt.FacetChart(...)

위의 모든 추가 코드를 고려할 때, 왜 명시적인 `facet` 연산자를 사용하고 싶을까요? 기본 차트의 경우 가능하다면 `column` 또는 `row` 인코딩 채널을 사용해야 합니다. 그러나 명시적으로 `facet` 연산자를 사용하는 것은 레이어 차트와 같이 구성된 뷰(composed views)를 패싯하고 싶을 때 유용합니다.

이전의 레이어 기온 플롯을 다시 살펴보겠습니다. 뉴욕과 시애틀의 데이터를 동일한 플롯에 플로팅하는 대신, 별도의 패싯으로 나누어 보겠습니다. 개별 차트 정의는 이전과 거의 동일합니다: 하나의 영역 차트와 하나의 선 차트입니다. 유일한 차이점은 이번에는 차트 생성자에 데이터를 직접 전달하지 않고, 나중에 패싯 연산자에 전달할 때까지 기다린다는 것입니다. 이전과 거의 동일하게 차트를 레이어로 쌓은 다음, 레이어 차트 객체에서 `facet`을 호출하여 데이터를 전달하고 `location` 필드를 기반으로 `column` 패싯을 지정할 수 있습니다:

In [18]:
tempMinMax = alt.Chart().mark_area(opacity=0.3).encode(
  alt.X('month(date):T', title=None, axis=alt.Axis(format='%b')),
  alt.Y('average(temp_max):Q', title='Avg. Temperature (°C)'),
  alt.Y2('average(temp_min):Q'),
  alt.Color('location:N')
)

tempMid = alt.Chart().mark_line().transform_calculate(
  temp_mid='(+datum.temp_min + +datum.temp_max) / 2'
).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_mid):Q'),
  alt.Color('location:N')
)

alt.layer(tempMinMax, tempMid).facet(
  data=weather,
  column='location:N'
)

alt.FacetChart(...)

지금까지 살펴본 패싯 차트는 패싯 셀 전체에서 동일한 축 스케일 도메인을 사용합니다. *공유(shared)* 스케일과 축을 사용하는 이 기본 설정은 값의 정확한 비교를 돕습니다. 그러나 셀의 값 범위가 크게 다른 경우와 같이 각 차트의 스케일을 독립적으로 지정하고 싶은 경우도 있을 수 있습니다.

레이어 차트와 마찬가지로 패싯 차트도 플롯 전체에서 독립적인 스케일이나 축으로 *해결(resolving)*하는 것을 지원합니다. `resolve_axis` 메서드를 호출하여 독립적인(`independent`) y축을 요청하면 어떤 일이 일어나는지 살펴봅시다:

In [19]:
tempMinMax = alt.Chart().mark_area(opacity=0.3).encode(
  alt.X('month(date):T', title=None, axis=alt.Axis(format='%b')),
  alt.Y('average(temp_max):Q', title='Avg. Temperature (°C)'),
  alt.Y2('average(temp_min):Q'),
  alt.Color('location:N')
)

tempMid = alt.Chart().mark_line().transform_calculate(
  temp_mid='(+datum.temp_min + +datum.temp_max) / 2'
).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_mid):Q'),
  alt.Color('location:N')
)

alt.layer(tempMinMax, tempMid).facet(
  data=weather,
  column='location:N'
).resolve_axis(y='independent')

alt.FacetChart(...)

*위의 차트는 거의 변하지 않은 것처럼 보이지만, 이제 시애틀에 대한 플롯에 자체 축이 포함되었습니다.*

대신 `resolve_scale`을 호출하여 기초가 되는 스케일 도메인을 해결하면 어떻게 될까요?

In [20]:
tempMinMax = alt.Chart().mark_area(opacity=0.3).encode(
  alt.X('month(date):T', title=None, axis=alt.Axis(format='%b')),
  alt.Y('average(temp_max):Q', title='Avg. Temperature (°C)'),
  alt.Y2('average(temp_min):Q'),
  alt.Color('location:N')
)

tempMid = alt.Chart().mark_line().transform_calculate(
  temp_mid='(+datum.temp_min + +datum.temp_max) / 2'
).encode(
  alt.X('month(date):T'),
  alt.Y('average(temp_mid):Q'),
  alt.Color('location:N')
)

alt.layer(tempMinMax, tempMid).facet(
  data=weather,
  column='location:N'
).resolve_scale(y='independent')

alt.FacetChart(...)

*이제 서로 다른 축 스케일 도메인을 가진 패싯 셀들이 보입니다. 이 경우에는 독립적인 스케일을 사용하는 것이 좋지 않은 생각인 것 같습니다! 도메인이 크게 다르지 않아서, 뉴욕과 시애틀의 최고 기온이 비슷하다고 오해할 수 있기 때문입니다.*

상투적인 표현을 빌리자면: 무언가를 할 수 있다고 해서 반드시 해야 한다는 뜻은 아닙니다...

## 결합 (Concatenate)

패싯은 데이터의 개별 하위 세트를 보여주는 [스몰 멀티플(small multiple)](https://en.wikipedia.org/wiki/Small_multiple) 플롯을 생성합니다. 그러나 우리는 *동일한* 데이터 세트의 서로 다른 뷰(하위 세트가 아님)나 *서로 다른* 데이터 세트가 포함된 뷰로 구성된 멀티 뷰 디스플레이를 만들고 싶을 수 있습니다.

Altair는 임의의 차트를 하나의 구성된 차트로 결합할 수 있는 *결합(concatenation)* 연산자를 제공합니다. `hconcat` 연산자(약칭 `|`)는 수평 결합을 수행하고, `vconcat` 연산자(약칭 `&`)는 수직 결합을 수행합니다.

이전에 보았던 것과 매우 유사하게, 뉴욕과 시애틀의 월별 평균 최고 기온을 보여주는 기본 선 차트부터 시작하겠습니다:

In [21]:
alt.Chart(weather).mark_line().encode(
  alt.X('month(date):T', title=None),
  alt.Y('average(temp_max):Q'),
  color='location:N'
)

alt.Chart(...)

*시간에 따른 기온뿐만 아니라 강수량과 풍속 수준도 비교하고 싶다면 어떻게 해야 할까요?*

세 개의 플롯으로 구성된 결합된 차트를 만들어 보겠습니다. 먼저 세 플롯이 공유해야 하는 모든 측면을 포함하는 "기본(base)" 차트 정의를 만드는 것부터 시작하겠습니다. 그런 다음 이 기본 차트를 수정하여 `temp_max`, `precipitation`, `wind` 필드에 대해 서로 다른 y축 인코딩을 가진 변형된 차트들을 만듭니다. 그런 다음 파이프(`|`) 약칭 연산자를 사용하여 이들을 수평으로 결합할 수 있습니다:

In [22]:
base = alt.Chart(weather).mark_line().encode(
  alt.X('month(date):T', title=None),
  color='location:N'
).properties(
  width=240,
  height=180
)

temp = base.encode(alt.Y('average(temp_max):Q'))
precip = base.encode(alt.Y('average(precipitation):Q'))
wind = base.encode(alt.Y('average(wind):Q'))

temp | precip | wind

alt.HConcatChart(...)

또는 파이프 `|` 연산자 대신 더 명시적인 `alt.hconcat()` 메서드를 사용할 수도 있습니다. *위의 코드를 `hconcat`을 사용하도록 다시 작성해 보세요.*

수직 결합은 수평 결합과 유사하게 작동합니다. *`&` 연산자(또는 `alt.vconcat` 메서드)를 사용하여 코드를 수정하여 수평 순서 대신 수직 순서를 사용하도록 하세요.*

마지막으로 수평 및 수직 결합을 결합할 수 있다는 점에 유의하세요. *`(temp | precip) & wind`와 같이 작성하면 어떤 일이 일어날까요?*

*참고*: 괄호의 중요성에 주목하세요... 괄호를 제거하면 어떻게 될까요? 이러한 오버로드된 연산자는 여전히 [파이썬의 연산자 우선순위 규칙](https://docs.python.org/3/reference/expressions.html#operator-precedence)을 따르므로, `&`를 사용한 수직 결합이 `|`를 사용한 수평 결합보다 우선순위가 높다는 점을 명심하세요!

나중에 다시 살펴보겠지만, 결합 연산자를 사용하면 모든 차트를 결합하여 멀티 뷰 대시보드를 만들 수 있습니다!

## 반복 (Repeat)

위의 결합 연산자는 매우 일반적이어서 임의의 차트를 구성할 수 있게 해줍니다. 그럼에도 불구하고 위의 예제는 여전히 다소 장황했습니다. 세 개의 매우 유사한 차트가 있음에도 불구하고 각각을 별도로 정의한 다음 결합해야 했기 때문입니다.

하나 또는 두 개의 변수만 변경되는 경우, `repeat` 연산자는 여러 차트를 만들기 위한 편리한 지름길을 제공합니다. 일부 자유 변수가 포함된 *템플릿(template)* 사양이 주어지면, 반복 연산자는 해당 변수에 지정된 각 할당에 대해 차트를 생성합니다.

`repeat` 연산자를 사용하여 위의 결합 예제를 다시 만들어 보겠습니다. 차트 간에 변경되는 유일한 측면은 `y` 인코딩 채널을 위한 데이터 필드 선택입니다. 템플릿 사양을 만들기 위해 *반복자 변수(repeater variable)* `alt.repeat('column')`을 y축 필드로 사용할 수 있습니다. 이 코드는 단순히 반복되는 차트들을 수평 방향으로 구성하는 `column` 반복자에 할당된 변수를 사용하겠다고 명시하는 것입니다. (반복자는 필드 이름만 제공하므로 필드 데이터 유형을 `type='quantitative'`로 별도로 지정해야 합니다.)

그런 다음 각 열에 대한 데이터 필드 이름을 전달하여 `repeat` 메서드를 호출합니다:

In [23]:
alt.Chart(weather).mark_line().encode(
  alt.X('month(date):T',title=None),
  alt.Y(alt.repeat('column'), aggregate='average', type='quantitative'),
  color='location:N'
).properties(
  width=240,
  height=180
).repeat(
  column=['temp_max', 'precipitation', 'wind']
)

alt.RepeatChart(...)

반복은 열과 행 모두에 대해 지원됩니다. *위의 코드를 `column` 대신 `row`를 사용하도록 수정하면 어떻게 될까요?*

우리는 `row`와 `column` 반복을 함께 사용할 수도 있습니다! 탐색적 데이터 분석을 위한 일반적인 시각화 중 하나는 [산점도 행렬(scatter plot matrix, 또는 SPLOM)](https://en.wikipedia.org/wiki/Scatter_plot#Scatterplot_matrices)입니다. 검사할 변수 컬렉션이 주어지면, SPLOM은 해당 변수들의 모든 쌍에 대한 플롯 그리드를 제공하여 잠재적인 연관성을 평가할 수 있게 해줍니다.

`repeat` 연산자를 사용하여 `temp_max`, `precipitation`, `wind` 필드에 대한 SPLOM을 만들어 보겠습니다. 먼저 x축과 y축 데이터 필드 모두에 대해 반복자 변수를 사용하여 템플릿 사양을 만듭니다. 그런 다음 `row`와 `column` 모두에 사용할 필드 이름 배열을 전달하여 `repeat`을 호출합니다. 그러면 Altair는 반복되는 차트의 전체 공간을 만들기 위해 [교차 곱(또는 데카르트 곱)](https://ko.wikipedia.org/wiki/%EA%B3%B1%EC%A7%91%ED%95%A9)을 생성합니다:

In [24]:
alt.Chart().mark_point(filled=True, size=15, opacity=0.5).encode(
  alt.X(alt.repeat('column'), type='quantitative'),
  alt.Y(alt.repeat('row'), type='quantitative')
).properties(
  width=150,
  height=150
).repeat(
  data=weather,
  row=['temp_max', 'precipitation', 'wind'],
  column=['wind', 'precipitation', 'temp_max']
).transform_filter(
  'datum.location == "Seattle"'
)

alt.RepeatChart(...)

*이 플롯들을 보면 강수량과 풍속 사이에는 강한 연관성이 없어 보이지만, 극단적인 풍속 및 강수량 이벤트가 유사한 기온 범위(~5-15° C)에서 발생한다는 것을 알 수 있습니다. 그러나 이 관찰은 특별히 놀랍지 않습니다. 패싯 섹션의 시작 부분에 있는 히스토그램을 다시 살펴보면, 최고 기온이 5-15° C 범위인 날이 가장 흔하게 발생한다는 것을 분명히 알 수 있기 때문입니다.*

*차트 반복을 더 잘 이해하기 위해 위의 코드를 수정해 보세요. SPLOM에 다른 변수(`temp_min`)를 추가해 보세요. `repeat` 연산자의 `row` 또는 `column` 매개변수에서 필드 이름의 순서를 바꾸면 어떤 일이 일어날까요?*

*마지막으로, `repeat` 연산자가 제공하는 기능을 진정으로 감상하기 위해, 오직 `hconcat`과 `vconcat`만을 사용하여 위의 SPLOM을 다시 만든다면 어떨지 잠시 상상해 보세요!*

## 뷰 결합 대수 (A View Composition Algebra)

결합 연산자인 `layer`, `facet`, `concat`, `repeat`은 함께 *뷰 구성 대수(view composition algebra)*를 형성합니다. 다양한 연산자를 결합하여 다양한 멀티 뷰 시각화를 구성할 수 있습니다.

예를 들어, 두 개의 기본 차트에서 시작해 보겠습니다: 히스토그램과 전체 평균을 보여주는 단순한 선(단일 `rule` 마크).

In [25]:
basic1 = alt.Chart(weather).transform_filter(
  'datum.location == "Seattle"'
).mark_bar().encode(
  alt.X('month(date):O'),
  alt.Y('average(temp_max):Q')
)

basic2 = alt.Chart(weather).transform_filter(
  'datum.location == "Seattle"'
).mark_rule(stroke='firebrick').encode(
  alt.Y('average(temp_max):Q')
)

basic1 | basic2

alt.HConcatChart(...)

그런 다음 `layer` 연산자를 사용하여 두 차트를 결합하고, 그 레이어된 차트를 `repeat`하여 여러 필드에 대해 평균이 겹쳐진 히스토그램을 보여줄 수 있습니다:

In [26]:
alt.layer(
  alt.Chart().mark_bar().encode(
    alt.X('month(date):O', title='월'),
    alt.Y(alt.repeat('column'), aggregate='average', type='quantitative')
  ),
  alt.Chart().mark_rule(stroke='firebrick').encode(
    alt.Y(alt.repeat('column'), aggregate='average', type='quantitative')
  )
).properties(
  width=200,
  height=150
).repeat(
  data=weather,
  column=['temp_max', 'precipitation', 'wind']
).transform_filter(
  'datum.location == "Seattle"'
)

alt.RepeatChart(...)

멀티 뷰 구성 연산자에만 집중하면 위의 시각화 모델은 다음과 같습니다:

```
repeat(column=[...])
|- layer
   |- basic1
   |- basic2
```

이제 시애틀 날씨에 대한 개요를 제공하는 최종 [대시보드(dashboard)](https://ko.wikipedia.org/wiki/%EB%8C%80%EC%8B%9C%EB%B3%B4%EB%93%9C_(%EB%B9%84%EC%A7%80%EB%8B%88%EC%8A%A4)) 내에서 *모든* 연산자를 적용하는 방법을 살펴보겠습니다. 이전 섹션의 SPLOM 및 패싯 히스토그램 디스플레이를 위의 반복된 히스토그램과 결합할 것입니다:

In [27]:
splom = alt.Chart().mark_point(filled=True, size=15, opacity=0.5).encode(
  alt.X(alt.repeat('column'), type='quantitative'),
  alt.Y(alt.repeat('row'), type='quantitative')
).properties(
  width=125,
  height=125
).repeat(
  row=['temp_max', 'precipitation', 'wind'],
  column=['wind', 'precipitation', 'temp_max']
)

dateHist = alt.layer(
  alt.Chart().mark_bar().encode(
    alt.X('month(date):O', title='월'),
    alt.Y(alt.repeat('row'), aggregate='average', type='quantitative')
  ),
  alt.Chart().mark_rule(stroke='firebrick').encode(
    alt.Y(alt.repeat('row'), aggregate='average', type='quantitative')
  )
).properties(
  width=175,
  height=125
).repeat(
  row=['temp_max', 'precipitation', 'wind']
)

tempHist = alt.Chart(weather).mark_bar().encode(
  alt.X('temp_max:Q', bin=True, title='최고 기온 (°C)'),
  alt.Y('count():Q'),
  alt.Color('weather:N', scale=alt.Scale(
    domain=['drizzle', 'fog', 'rain', 'snow', 'sun'],
    range=['#aec7e8', '#c7c7c7', '#1f77b4', '#9467bd', '#e7ba52']
  ))
).properties(
  width=115,
  height=100
).facet(
  column='weather:N'
)

alt.vconcat(
  alt.hconcat(splom, dateHist),
  tempHist,
  data=weather,
  title='시애틀 날씨 대시보드'
).transform_filter(
  'datum.location == "Seattle"'
).resolve_legend(
  color='independent'
).configure_axis(
  labelAngle=0
)

alt.VConcatChart(...)

이 대시보드의 전체 구성 모델은 다음과 같습니다:

```
vconcat
|- hconcat
|  |- repeat(row=[...], column=[...])
|  |  |- splom 기본 차트
|  |- repeat(row=[...])
|     |- layer
|        |- dateHist 기본 차트 1
|        |- dateHist 기본 차트 2
|- facet(column='weather')
   |- tempHist 기본 차트
```

*휴!* 대시보드에는 레이아웃을 개선하기 위한 몇 가지 사용자 정의도 포함되어 있습니다:

- 정렬을 돕고 전체 시각화가 화면에 맞도록 차트의 `width` 및 `height` 속성을 조정합니다.
- 색상 범례가 기온별 색상 히스토그램과 직접 연결되도록 `resolve_legend(color='independent')`를 추가합니다. 그렇지 않으면 범례가 대시보드 전체로 해결됩니다.
- 축 레이블이 회전하지 않도록 `configure_axis(labelAngle=0)`를 사용합니다. 이는 SPLOM의 산점도와 오른쪽의 월별 히스토그램 사이의 적절한 정렬을 보장하는 데 도움이 됩니다.

*이러한 조정 사항을 제거하거나 수정해보고 대시보드 레이아웃이 어떻게 반응하는지 확인해 보세요!*

이 대시보드는 다른 위치나 다른 데이터 세트의 데이터를 보여주기 위해 재사용될 수 있습니다. *시애틀 대신 뉴욕의 날씨 패턴을 보여주도록 대시보드를 업데이트해 보세요.*

## 요약 (Summary)

서브플롯 간격 및 헤더 레이블 제어를 포함하여 멀티 뷰 구성에 대한 자세한 내용은 [Altair 복합 차트(Compound Charts) 문서](https://altair-viz.github.io/user_guide/compound_charts.html)를 참조하세요.

이제 여러 뷰를 구성하는 방법을 살펴보았으므로, 이를 실제로 적용할 준비가 되었습니다. 여러 뷰는 데이터를 정적으로 표현하는 것 외에도 대화형 다차원 탐색을 가능하게 할 수 있습니다. 예를 들어, *연결된 선택(linked selections)*을 사용하여 한 뷰에서 점을 강조 표시하면 다른 뷰에서 해당 값이 강조 표시되도록 할 수 있습니다.

다음 노트북에서는 개별 플롯과 멀티 뷰 구성 모두에 대해 *대화형 선택*을 작성하는 방법을 살펴보겠습니다.